# Validation A — calculated emittance

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/radcoolpv-py/blob/main/docs/site/notebooks/validation_a_optics.ipynb)

**Akerboom *et al.*, *ACS Photonics* 9 (2022) 3831–3840,
[doi:10.1021/acsphotonics.2c01389](https://doi.org/10.1021/acsphotonics.2c01389)**

## The physics

A silicon module radiates heat to the sky through the 8–13 µm atmospheric
window. Bare silicon under a gold back reflector barely emits there at all, so
it runs hot. Silica does emit strongly in that band — the Si–O stretching
resonance — so a silica layer turns the module into a thermal emitter.

Patterning that silica into microcylinders adds a second effect. The array is
comparable in size to the wavelength, so it behaves as a graded index between
air and silica, suppressing reflection and letting the surface emit closer to a
blackbody across a wider band.

This group computes emittance directly from the geometry with RCWA. The gold
blocks transmission, so $\epsilon = 1 - R$, and Kirchhoff's law lets the
absorptance be read as emittance.

Silicon is modeled as **nonabsorbing** here — the paper's own stated assumption
for the cooling band.

## Main result

Mean emittance over 7.5–16 µm, against the paper's Figure 3a:

| Surface | radcoolpv | Digitized | Paper text |
|---|---:|---:|---:|
| Bare Au/Si | 0.032 | 0.036 | ~3.5% |
| Flat silica | 0.842 | 0.843 | — |
| Silica cylinders | 0.984 | 0.977 | — |

The optics agree. The cylinders emit almost as a blackbody; the bare module is
essentially transparent to its own heat.

## Set up the runtime

Colab runtimes are temporary. Run this again after a reset.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from IPython.display import Markdown, display

PROJECT = Path("/content/radcoolpv-py")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/gsilvaoelker/radcoolpv-py.git", str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--editable", "."],
               cwd=PROJECT, check=True)
os.chdir(PROJECT)

from radcoolpv import config, pipeline, report
print("radcoolpv ready in", PROJECT)

## Build the solver

This group computes the optics from the geometry, so it needs S4 — a C++
extension with no PyPI package, built from source in about ten minutes.

In [ ]:
S4_COMMIT = "9569f5e555b967a4324eb1ea593d0f9f40761a61"   # the tested revision

def build_s4():
    import importlib, importlib.util
    if importlib.util.find_spec("S4") is not None:
        print("S4 is already importable."); return
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "build-essential", "git",
                    "libboost-all-dev", "libfftw3-dev", "liblapack-dev",
                    "libopenblas-dev", "libsuitesparse-dev"], check=True)
    src = Path("/content/S4")
    if not src.exists():
        subprocess.run(["git", "clone", "https://github.com/phoebe-p/S4.git", str(src)], check=True)
    subprocess.run(["git", "checkout", S4_COMMIT], cwd=src, check=True)
    subprocess.run(["make", "-j2", "S4_pyext"], cwd=src, check=True)
    importlib.invalidate_caches()

build_s4()

## The case

This is the paper's structure, written out in full. Edit anything: the cylinder
radius and height, the pitch, the layer thicknesses, the wavelength range.

`lattice.x` is `6.125 × √3`, because radcoolpv builds a hexagonal lattice as a
centered-rectangular cell with a second cylinder at `(x/2, y/2)` — that is only
hexagonal when `x = y√3`.

`n` and `s4_modes` are reduced below so the cell finishes in a few minutes. The
converged values are `n: 281` and `s4_modes: 60`; raise them and check the
emittance has stopped moving before quoting a number.

In [ ]:
%%writefile validation_a.yaml
run:
  optics: true
  thermal: false
  plots: true
  mode: standard
  write_outputs: true
  results_dir: results/validation_a

simulation:
  wavelength: {min: 2.0, max: 16.0, n: 60}    # converged: 281
  angles: normal
  polarization: unpolarized
  s4_modes: 20                                # converged: 60

geometry:
  source: s4
  shape: cylinder               # flat -> the unpatterned silica reference
  photonic_material: sio2       # vacuum -> the bare Au/Si reference
  lattice: {type: hexagonal, x: 10.608811, y: 6.125}
  cylinder: {radius: 1.75, height: 2.25}

structure:
  - {material: sio2, thickness: 500.0}
  - {material: silicon, thickness: 500.0}
  - {material: gold, thickness: 0.08}
  - {material: vacuum, thickness: 0.0, terminal: true}

materials:
  sio2: PalikKitamura_SiO2
  silicon: Akerboom_Si_lossless
  gold: RII_Olmon_2012_ev_Au

comparison:
  spectra:
    - {label: Paper Fig. 3a, file: validation/data/fig3a_calculated_emittance.txt,
       column: 3, color: '#0000ff'}

In [ ]:
CASE = "validation_a.yaml"
ctx = pipeline.run(config.load_cases(CASE)[0])
report.summary(ctx)

## Check the band average

`band_average` integrates exactly the band asked for, so this does not depend
on where the grid samples fall.

In [ ]:
import numpy as np
from radcoolpv.optics.averages import band_average

lam, emit = ctx.optics.lambda_um, ctx.optics.emit
print(f"mean emittance 7.5-16 um: {band_average(lam, emit, 7.5, 16.0):.4f}"
      f"   (converged: 0.984, paper: 0.977)")
print(f"mean emittance 8-13 um  : {band_average(lam, emit, 8.0, 13.0):.4f}")

**Exercise.** Set `photonic_material: vacuum` with `shape: flat` and the three-layer stack to get the bare module, and compare. How much of the emittance is the silica, and how much is the patterning?

## Use your own material

This case computes the optics from the geometry, so what it needs from you is
an **optical constants table**, not a spectrum. Upload a CSV whose first line is
`lambda_um,n,k`: wavelength in micrometres, then the real and imaginary parts of
the refractive index.

The filename is the model name. `MyGlass.csv` becomes usable as `MyGlass` in the
`materials:` block above, so you can swap it in for `sio2` and re-run. Its
wavelength range has to cover the range the case asks for — the loader raises
rather than extrapolating past the last tabulated point.


In [ ]:
from google.colab import files
from radcoolpv.materials import registry

MATERIALS = PROJECT / "radcoolpv" / "materials" / "data"

for name, blob in files.upload().items():
    header = blob[:200].decode(errors="ignore").splitlines()[0].strip().lower()
    if name.lower().endswith(".csv") and header.startswith("lambda_um,"):
        (MATERIALS / name).write_bytes(blob)
        print(f"{name}: installed as material {Path(name).stem!r}")
    else:
        print(f"{name}: not installed. Expected a .csv whose first line is "
              f"'lambda_um,n,k'; this one starts {header[:40]!r}")

print("\navailable materials:", ", ".join(sorted(registry.available())))
